### Import libraries

In [24]:
import pandas as pd
import numpy as np
import os
import math
import seaborn as sns
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import GofChisquarePower
from sklearn.cluster import KMeans
import sklearn.metrics as sil
from ratio import find_power_vals

### Import humans

In [25]:
humans = pd.read_pickle('./data/phenotype/humans_maf_popstrat.pkl')

### Looping

In [26]:
maf_pops = []
for f in list(humans.columns):
    try:
        f.split("pheno_maf_")[1]
        maf_pops.append(f.split("pheno_maf_")[1])
    except:
        pass

In [27]:
df_chrom = []
df_file_name = []
df_snp_name = []
df_maf = []
df_maf_pop = []
df_intercept = []
df_effect = []
df_p_intercept = []
df_p_effect = []
df_SE = []
df_power = []

In [28]:
df_tot1 = pd.read_pickle("./data/effect_pop_strat1.pkl")
df_tot2 = pd.read_pickle("./data/effect_pop_strat2.pkl")
df_tot3 = pd.read_pickle("./data/effect_pop_strat3.pkl")

In [29]:
chrom = 21

In [30]:
len(os.listdir(f"./data/genotype/chrom_{chrom}"))

130

In [31]:
onlyfiles_input = [f for f in os.listdir(f"./data/genotype/chrom_{chrom}")]
onlyfiles_output1 = [f for f in list(set(df_tot1['file_name']))]
onlyfiles_output2 = [f for f in list(set(df_tot2['file_name']))]
onlyfiles_output3 = [f for f in list(set(df_tot3['file_name']))]

temp = [x for x in onlyfiles_input if x not in onlyfiles_output1]
temp2 = [x for x in temp if x not in onlyfiles_output2]
onlyfiles = [x for x in temp2 if x not in onlyfiles_output3]

In [32]:
ohm = len(onlyfiles_output1) + len(onlyfiles_output2)
chroms = [21]
for chrom in chroms:
    for snps in onlyfiles:
        print(f"advancement for chrom {chrom} = {round((ohm/len(onlyfiles_input))*100,2)}%")
        path_snp = f"./data/genotype/chrom_{chrom}/{snps}"
        
        geno = pd.read_pickle(path_snp)
        for i in range(len(geno.columns)):
            snp_num = i
            snp_name = geno.columns[snp_num]
            geno[geno.columns[snp_num]].value_counts()
            num_maj = geno[geno.columns[snp_num]].value_counts()[1]
            num_het = geno[geno.columns[snp_num]].value_counts()[0]
            try:
                num_min = geno[geno.columns[snp_num]].value_counts()[-1]
            except Exception as e:
                num_min = 0
            total_humans = num_maj + num_het + num_min
            maf = (num_min*2 + num_het)/(total_humans*2)
            min_power = 80
            [a, b, c] = find_power_vals(min_power)
            ratio = a*maf**2 + b*maf + c

            true_mean = 0
            true_effect = 5
            true_sd = true_effect*ratio

            mean_min = true_mean
            mean_het = true_mean
            mean_maj = true_mean
            sigma_min = true_sd
            sigma_het = true_sd
            sigma_maj = true_sd

            for maf_pop in maf_pops:
                humans[f"causal"] = geno[geno.columns[snp_num]]
                humans['pheno'] = humans[f"pheno_maf_{maf_pop}"]
                X = humans[['causal']]
                y = humans[['pheno']]
                reg = LinearRegression().fit(X, y)
                estimated_effect = reg.coef_[0][0]
                estimated_intercept = reg.intercept_[0]
                N = len(X)
                p = len(X.columns) + 1

                X_with_intercept = np.empty(shape=(N, p), dtype=float)
                X_with_intercept[:, 0] = 1
                X_with_intercept[:, 1:p] = X.values

                params = np.append(reg.intercept_,reg.coef_)
                predictions = reg.predict(X)
                new_y = (y-predictions)**2
                newX = pd.DataFrame({"Constant":np.ones(len(X))}).join(pd.DataFrame(list(X['causal'])))
                MSE = (sum(list(new_y['pheno'])))/(len(newX)-len(newX.columns))

                var_b = MSE*(np.linalg.inv(np.dot(newX.T,newX)).diagonal())
                sd_b = np.sqrt(var_b)
                ts_b = params/ sd_b
                p_values =[2*(1-stats.t.cdf(np.abs(i),(len(newX)-len(newX.columns)))) for i in ts_b]

                residuals = y - predictions
                residual_sum_of_squares = residuals.T @ residuals
                sigma_squared_hat = residual_sum_of_squares['pheno'].pheno / (N - p)
                var_beta_hat = np.linalg.inv(X_with_intercept.T @ X_with_intercept) * sigma_squared_hat
                SE = []
                for p_ in range(p):
                    standard_error = var_beta_hat[p_, p_] ** 0.5
                    SE.append(standard_error)

                sd_b = np.round(sd_b,3)
                ts_b = np.round(ts_b,3)
                params = np.round(params,4)

                distribution_nul = np.random.normal(0, SE[1], 5000)
                distribution_alt = np.random.normal(estimated_effect, SE[1], 5000)

                dis_only_alt = pd.DataFrame(data={"Z": list(distribution_alt)})
                dis_only_alt['Z2'] = (dis_only_alt['Z']/SE[1])**2
                cuttof = stats.chi2.ppf(0.8, df=1)
                omega = 0
                for val in list(dis_only_alt['Z2']):
                    if val > cuttof:
                        omega += 1
                    else:
                        pass
                power = omega/len(list(dis_only_alt['Z2']))
                #print(f"chrom = {chrom}")
                #print(f"file name = {snps}")
                #print(f"snp name = {snp_name}")
                #print(f"maf = {maf}")
                #print(f"strat distribution = {maf_pop}")
                #print(f"estimated intercept = {estimated_intercept}")
                #print(f"estimated effect = {estimated_effect}")
                #print(f"p value intercept = {p_values[0]}")
                #print(f"p value effect = {p_values[1]}")
                #print(f"standard error = {SE[1]}")
                #print(f"power = {round(power*100,8)}")
                
                df_chrom.append(chrom)
                df_file_name.append(snps)
                df_snp_name.append(snp_name)
                df_maf.append(maf)
                df_maf_pop.append(maf_pop)
                df_intercept.append(estimated_intercept)
                df_effect.append(estimated_effect)
                df_p_intercept.append(p_values[0])
                df_p_effect.append(p_values[1])
                df_SE.append(SE[1])
                df_power.append(power)
            i += 5
    ohm += 1

df_tot = pd.DataFrame(data = {'chrom':df_chrom,
                              'file_name':df_file_name,
                              'snp_name': df_snp_name,
                              'maf':df_maf,
                              'maf_pop': df_maf_pop,
                              'intercept':df_intercept,
                              'effect':df_effect,
                              'p_intercept': df_p_intercept,
                              'p_effect': df_p_effect,
                              'SE': df_SE,
                              'power':df_power
                             })
df_tot.to_pickle("./data/effect_pop_strat4.pkl")

advancement for chrom 21 = 73.08%


/tmp/ipykernel_841/2891524639.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  humans[f"causal"] = geno[geno.columns[snp_num]]
/tmp/ipykernel_841/2891524639.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  humans['pheno'] = humans[f"pheno_maf_{maf_pop}"]


advancement for chrom 21 = 73.08%
advancement for chrom 21 = 73.08%
advancement for chrom 21 = 73.08%
advancement for chrom 21 = 73.08%


KeyboardInterrupt: 

In [ ]:
df_tot.sort_values(by=['effect'])

In [ ]:
df_tot['effectsquared'] = df_tot['effect']**2

In [ ]:
df_tot.sort_values(by=['effectsquared'])

In [33]:
df_tot = pd.DataFrame(data = {'chrom':df_chrom,
                              'file_name':df_file_name,
                              'snp_name': df_snp_name,
                              'maf':df_maf,
                              'maf_pop': df_maf_pop,
                              'intercept':df_intercept,
                              'effect':df_effect,
                              'p_intercept': df_p_intercept,
                              'p_effect': df_p_effect,
                              'SE': df_SE,
                              'power':df_power
                             })
df_tot.to_pickle("./data/effect_pop_strat4.pkl")

In [34]:
df_tot

,chrom,file_name,snp_name,maf,maf_pop,intercept,effect,p_intercept,p_effect,SE,power
0,21,6_1_1200_6_maf_0.1_0.13.pkl,21:23761045:C:T_C,0.112420,0.01_0,-0.380151,0.080415,0.000000e+00,6.613166e-04,0.023586,0.9828
1,21,6_1_1200_6_maf_0.1_0.13.pkl,21:23761045:C:T_C,0.112420,0.01_1,-0.266990,-0.075674,0.000000e+00,4.013579e-05,0.018394,0.9988
2,21,6_1_1200_6_maf_0.1_0.13.pkl,21:23761045:C:T_C,0.112420,0.01_2,-0.155823,0.022187,0.000000e+00,3.018102e-03,0.007473,0.9554
3,21,6_1_1200_6_maf_0.1_0.13.pkl,21:23761045:C:T_C,0.112420,0.01_3,0.461898,-0.001948,0.000000e+00,9.145819e-01,0.018161,0.2086
4,21,6_1_1200_6_maf_0.1_0.13.pkl,21:23761045:C:T_C,0.112420,0.01_4,-0.059952,-0.064793,8.730122e-08,2.143771e-07,0.012458,1.0000
...,...,...,...,...,...,...,...,...,...,...,...
601368,21,0_5_1200_0_maf_0.01_0.016.pkl,21:18535815:A:C_A,0.011382,0.32_8,-1.653844,1.869955,1.893596e-12,3.774758e-15,0.236274,1.0000
601369,21,0_5_1200_0_maf_0.01_0.016.pkl,21:18535815:A:C_A,0.011382,0.32_9,0.122737,0.016006,5.198376e-01,9.338464e-01,0.192807,0.2078
601370,21,0_5_1200_0_maf_0.01_0.016.pkl,21:18535815:A:C_A,0.011382,0.38_0,0.005553,-0.123740,9.758340e-01,5.044297e-01,0.185342,0.2896
601371,21,0_5_1200_0_maf_0.01_0.016.pkl,21:18535815:A:C_A,0.011382,0.38_1,0.496524,-0.407067,1.025065e-02,3.734704e-02,0.195418,0.7900
